# Kandinsky V2A — Son 5 Saniyede Seed Karşılaştırması

Bu deney 15 saniyelik videonun **10–15. saniyeleri** için üç aday üretir. Video, prompt, Pro Distill modeli ve ayarlar aynı kalır; yalnızca seed değişir. Seed 42 önceki sonuçla karşılaştırma sağlar; 123 ve 7 yeni adaylardır.

Bu bir eğitim değil, hazır modelle üretimdir. Adaylar sırayla çalışır. Önceki pencerelerde kullandığın prompt'u aynen koru.

Akış: GPU/disk kontrolü → Kandinsky ortamını kurma → videoyu ve prompt'u seçme → üç adayı sırayla üretip dinleme → rapor ZIP'ini indirme.

## 1 — GPU ve disk kontrolü

Colab'de A100 80 GB seç. Pro Distill için en az 90 GiB boş disk gerekir. Sistem RAM'i ve GPU belleği farklı kaynaklardır. Her adayın üretimi boyunca ikisini de örnekleyip rapora kaydedeceğiz.

In [ ]:
import os, sys, shutil, subprocess, json, time, hashlib, zipfile, threading
from pathlib import Path

print("Python:", sys.version)
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, check=False
    )
except FileNotFoundError:
    raise RuntimeError("Colab Runtime ayarlarından GPU seçip yeniden bağlan.")
print(gpu.stdout or gpu.stderr)
if gpu.returncode != 0:
    raise RuntimeError("GPU denetlenemedi.")
if sys.version_info < (3, 13):
    raise RuntimeError("Kandinsky pipeline Python 3.13 veya 3.14 gerektirir.")
gpu_line = gpu.stdout.strip().splitlines()[0]
if "A100" not in gpu_line or "80" not in gpu_line:
    raise RuntimeError("Bu Pro Distill deneyi A100 80 GB için hazırlandı.")
free_gib = shutil.disk_usage("/content").free / (1024**3)
print(f"/content boş disk: {free_gib:.1f} GiB")
if free_gib < 90:
    raise RuntimeError("En az 90 GiB boş disk alanı gerekli.")

## 2 — Resmî ortamı kur ve Pro Distill'i indir

Bu adım kaynak kodunu ve model ağırlıklarını indirir. Colab runtime'ı sıfırlanmadıysa ağırlıklar önbellekte bulunabilir.

In [ ]:
%cd /content
!git clone --depth 1 https://github.com/kandinskylab/kandinsky-6.git
%cd /content/kandinsky-6
!python -m pip -q install uv psutil
!curl -fsSL https://just.systems/install.sh | bash -s -- --to /usr/local/bin
!just setup
print("Kaynak commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
!just download pro-distill
print("Pro Distill hazır.")

## 3 — Videoyu ve sabit koşulları seç

15 saniyelik kaynak videonun son 5 saniyesini deniyoruz; pencere başlangıcı 10.0 saniye. Prompt'u önceki pencerelerde kullandığın ifadeyle aynı bırak. Bu deneyde yalnızca seed değişecek.

In [ ]:
from google.colab import files
from IPython.display import display, Video, Audio, Markdown

EVENT_PROMPT = "footsteps"  # Önceki pencerelerde kullandığın prompt ile aynı olmalı.
WINDOW_START_SECONDS = 10.0
GENERATION_SECONDS = 5
SEEDS = [42, 123, 7]

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Bir MP4 video yükle.")
filename = next(iter(uploaded))
source_video = Path("/content") / Path(filename).name
source_video.write_bytes(uploaded[filename])

probe = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries", "format=duration",
     "-of", "default=noprint_wrappers=1:nokey=1", str(source_video)],
    capture_output=True, text=True
)
if probe.returncode:
    raise RuntimeError("Video okunamadı: " + probe.stderr[-1200:])
source_duration = float(probe.stdout.strip())
window_end = WINDOW_START_SECONDS + GENERATION_SECONDS
print(f"Video {source_duration:.2f} sn | pencere {WINDOW_START_SECONDS:.1f}–{window_end:.1f} sn")
print("Prompt:", EVENT_PROMPT, "| Seed'ler:", SEEDS)
if WINDOW_START_SECONDS < 0 or window_end > source_duration + 0.02:
    raise ValueError("Seçilen beş saniyelik pencere kaynak videonun dışında.")

input_video = Path("/content/kandinsky_seed_sweep_input.mp4")
prep = subprocess.run(
    ["ffmpeg", "-y", "-ss", str(WINDOW_START_SECONDS), "-i", str(source_video),
     "-t", str(GENERATION_SECONDS), "-map", "0:v:0", "-an",
     "-c:v", "libx264", "-pix_fmt", "yuv420p", "-r", "24", str(input_video)],
    capture_output=True, text=True
)
if prep.returncode:
    raise RuntimeError("Video kesiti hazırlanamadı: " + prep.stderr[-1500:])
input_sha256 = hashlib.sha256(input_video.read_bytes()).hexdigest()
display(Video(data=input_video.read_bytes(), embed=True, mimetype="video/mp4"))

## 4 — Üç adayı sırayla üret

Her aday aynı 5 saniyelik kesiti, prompt'u ve Pro Distill ayarlarını kullanır. RAM ve GPU belleği yaklaşık 1.5 saniyede bir örneklenir. Üretimler paralel çalışmaz.

In [ ]:
import psutil

DEVICE_CONFIG = Path("/content/kandinsky-6/kandinsky/configs/devices/a100-pro-distill-sdpa.yaml")
DEVICE_CONFIG.write_text(
    "checkpoint: pro-distill\ncache:\n  mode: none\noffload:\n  strategy: module\n"
    "compile:\n  strategy: none\nattention:\n  engine: sdpa\n"
    "text_embedder:\n  quantized_qwen: false\nbeautifier:\n  name: none\n",
    encoding="utf-8",
)
RUN_DIR = Path("/content/kandinsky_seed_sweep")
RUN_DIR.mkdir(parents=True, exist_ok=True)
candidates = []

def run_and_sample_memory(command):
    box = {}
    def invoke():
        box["result"] = subprocess.run(
            command, cwd="/content/kandinsky-6", text=True, capture_output=True
        )
    worker = threading.Thread(target=invoke, daemon=True)
    worker.start()
    peak_ram = 0.0
    peak_vram = 0.0
    while worker.is_alive():
        try:
            peak_ram = max(peak_ram, psutil.virtual_memory().used / (1024**3))
        except Exception:
            pass
        try:
            sample = subprocess.run(
                ["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                capture_output=True, text=True, check=True
            )
            peak_vram = max(peak_vram, float(sample.stdout.strip().splitlines()[0]) / 1024)
        except Exception:
            pass
        worker.join(timeout=1.5)
    worker.join()
    return box["result"], peak_ram, peak_vram

for seed in SEEDS:
    output_mp4 = RUN_DIR / f"candidate_seed_{seed}.mp4"
    output_wav = RUN_DIR / f"candidate_seed_{seed}.wav"
    command = [
        "just", "generate", EVENT_PROMPT, "--config", str(DEVICE_CONFIG),
        "--attention-engine", "sdpa", "--video-file", str(input_video),
        "--time-length", str(GENERATION_SECONDS), "--out", str(output_mp4),
        "--seed", str(seed),
    ]
    print(f"\nSeed {seed}: üretim başlıyor.")
    started = time.time()
    result, peak_ram, peak_vram = run_and_sample_memory(command)
    elapsed = time.time() - started
    print(result.stdout[-3000:])
    if result.returncode:
        print(result.stderr[-5000:])
        raise RuntimeError(f"Seed {seed} üretimi başarısız oldu.")
    if not output_mp4.is_file():
        raise FileNotFoundError(f"Çıktı bulunamadı: {output_mp4}")
    extract = subprocess.run(
        ["ffmpeg", "-y", "-i", str(output_mp4), "-vn", "-ac", "2", "-ar", "44100", str(output_wav)],
        capture_output=True, text=True
    )
    if extract.returncode:
        raise RuntimeError(f"WAV çıkarılamadı: {extract.stderr[-1200:]}")
    candidates.append({
        "seed": seed, "video_file": output_mp4.name, "audio_file": output_wav.name,
        "generation_seconds": round(elapsed, 2),
        "peak_system_ram_gib_sampled": round(peak_ram, 2),
        "peak_gpu_memory_gib_sampled": round(peak_vram, 2),
    })
    print(f"Seed {seed}: {elapsed:.1f} sn | RAM {peak_ram:.1f} GiB | GPU {peak_vram:.1f} GiB")
print("Üç aday tamamlandı.")

## 5 — Adayları dinle ve karşılaştır

Olayla eşleşme, doğallık, gürültü/bozulma ve önceki parçalarla uyuma bak. Seed farklılığı yeni bir örnekleme sağlar; tek başına doğru olay sesini garanti etmez.

In [ ]:
for item in candidates:
    display(Markdown(
        f"### Seed {item['seed']} — {item['generation_seconds']:.1f} sn"
        f" | RAM {item['peak_system_ram_gib_sampled']:.1f} GiB"
        f" | GPU {item['peak_gpu_memory_gib_sampled']:.1f} GiB"
    ))
    display(Audio(filename=str(RUN_DIR / item["audio_file"])))

## 6 — Rapor ZIP'ini indir

ZIP içinde üç adayın MP4 ve WAV dosyaları, ayarlar ve her üretimin süre/RAM/VRAM ölçümleri yer alır.

In [ ]:
report = {
    "experiment": "kandinsky_v2a_last_window_seed_sweep_v1",
    "checkpoint": "kandinskylab/Kandinsky-6.0-Pro-distill-5s-Diffusers",
    "source_repo": "https://github.com/kandinskylab/kandinsky-6",
    "source_commit": subprocess.check_output(
        ["git", "-C", "/content/kandinsky-6", "rev-parse", "HEAD"], text=True
    ).strip(),
    "input_video_name": filename,
    "input_video_sha256": input_sha256,
    "prompt": EVENT_PROMPT,
    "prompt_expansion": "disabled",
    "window_start_seconds": WINDOW_START_SECONDS,
    "duration_seconds": GENERATION_SECONDS,
    "seeds": SEEDS,
    "gpu": gpu.stdout.strip(),
    "candidates": candidates,
    "ratings": {
        "best_seed": None, "event_match_1_to_5": None,
        "naturalness_1_to_5": None, "noise_or_artifacts_1_to_5": None, "notes": ""
    },
    "interpretation_limit": "Three candidates from one window do not establish a general success rate."
}
report_path = RUN_DIR / "report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
bundle_path = Path("/content/kandinsky_last_window_seed_sweep.zip")
with zipfile.ZipFile(bundle_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in candidates:
        archive.write(RUN_DIR / item["video_file"], arcname=item["video_file"])
        archive.write(RUN_DIR / item["audio_file"], arcname=item["audio_file"])
    archive.write(report_path, arcname="report.json")
print(json.dumps(report, ensure_ascii=False, indent=2))
files.download(str(bundle_path))